# Demo: End-to-End Context Understanding from Music (GNN-BERT)
This notebook demonstrates the end-to-end inference pipeline for **GNN-Based BERT for Understanding Context from Music**.

### Pipeline Steps:
1. Load an audio track (.wav)
2. Extract log-mel spectrogram (128 bins) and chroma features (12 bins)
3. Segment the track and construct a relational music structure graph
4. Tokenize accompanying text prompt / lyrics / description
5. Execute GNN-BERT cross-attention fusion
6. Display predicted multi-label tags (genre/mood) and DEAM emotion values (valence/arousal)
7. Demonstrate cross-modal text-to-audio retrieval

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add src to python path
sys.path.append(os.path.abspath("../src"))

from audio_features import AudioFeatureExtractor
from graph_builder import MusicGraphBuilder
from fusion_model import MusicGNNBERTFusionModel
from contrastive import DualEncoderGNNBERT

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Feature Extraction & Graph Construction
We load a sample audio track, compute 128-bin log-mel spectrogram and 12-bin chroma features, and build the segment graph.

In [ ]:
extractor = AudioFeatureExtractor(sample_rate=22050, segment_duration=3.0)
builder = MusicGraphBuilder(similarity_threshold_tau=0.60)

# Use track_0000.wav or synthesize an on-the-fly sample
audio_path = "../data/raw/audio/track_0000.wav"
if os.path.exists(audio_path):
    audio_y, sr = extractor.load_audio(audio_path)
else:
    from sample_data_gen import synthesize_audio_track
    audio_y = synthesize_audio_track(duration=12.0, genre="electronic", mood="energetic")
    sr = 22050

mel = extractor.extract_mel_spectrogram(audio_y)
chroma = extractor.extract_chroma(audio_y)
segments = extractor.segment_track(audio_y)
seg_features = extractor.extract_segment_features(audio_y, segments)
graph = builder.build_segment_graph(seg_features, track_id="demo_track")

print(f"Audio length: {len(audio_y)/sr:.2f}s")
print(f"Log-mel spectrogram shape: {mel.shape}")
print(f"Chroma features shape: {chroma.shape}")
print(f"Segment graph nodes: {graph.x.shape[0]}, edges: {graph.edge_index.shape[1]}")

## 2. Visualize Spectrogram and Relational Graph

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6))
axs[0].imshow(mel, aspect='auto', origin='lower', cmap='inferno')
axs[0].set_title("128-bin Log-Mel Spectrogram")
axs[0].set_ylabel("Mel Bins")

axs[1].imshow(chroma, aspect='auto', origin='lower', cmap='coolwarm')
axs[1].set_title("12-bin Chroma Pitch Class Features")
axs[1].set_ylabel("Pitch Classes")
axs[1].set_xlabel("Time Frames")
plt.tight_layout()
plt.show()

## 3. End-to-End Multi-Modal Context Inference
Combine the audio structure graph with textual context to predict multi-label tags and emotion scores.

In [ ]:
TAG_VOCAB = [
    "rock", "jazz", "electronic", "hiphop", "classical", "folk", "pop", "ambient",
    "energetic", "calm", "melancholic", "happy", "dark", "uplifting",
    "guitar", "piano", "drums", "synthesizer", "strings", "bass"
]

# Instantiate fusion model
model = MusicGNNBERTFusionModel(num_classes=len(TAG_VOCAB), fusion_type="cross_attention").to(device)
model.eval()

caption = "An energetic electronic dance track featuring pulsing synthesizer riffs and high tempo percussion."
print("Input text prompt:", caption)

# Tokenize
tokens = [min(255, ord(c)) for c in caption[:128]]
tokens += [0] * (128 - len(tokens))
input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
attn_mask = torch.ones((1, 128), dtype=torch.long, device=device)

# Forward pass
with torch.no_grad():
    out = model(
        x=graph.x.to(device),
        edge_index=graph.edge_index.to(device),
        batch=torch.zeros(graph.x.size(0), dtype=torch.long, device=device),
        input_ids=input_ids,
        attention_mask=attn_mask,
    )

probs = out["probs"].squeeze(0).cpu().numpy()
pred_valence = out["pred_valence"].item()
pred_arousal = out["pred_arousal"].item()

top_indices = np.argsort(-probs)[:5]
print("\n--- Predicted Top Context Tags ---")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. {TAG_VOCAB[idx]}: {probs[idx]:.4f}")

print(f"\n--- Predicted Continuous Emotion (DEAM) ---")
print(f"Valence (1.0 to 9.0): {pred_valence:.2f}")
print(f"Arousal (1.0 to 9.0): {pred_arousal:.2f}")

## 4. Cross-Modal Text-to-Audio Retrieval
Query the audio library using natural language captions via the InfoNCE dual-encoder.

In [ ]:
dual_encoder = DualEncoderGNNBERT().to(device)
dual_encoder.eval()

with torch.no_grad():
    g_embed = dual_encoder.encode_graph(graph.x.to(device), graph.edge_index.to(device), None)
    t_embed = dual_encoder.encode_text(input_ids, attn_mask)
    sim = torch.dot(g_embed.squeeze(0), t_embed.squeeze(0)).item()

print(f"Cross-Modal Embedding Similarity Score: {sim:.4f}")